# Data understanding — EV charge sessions

Exploration only. Nothing here is part of the pipeline: every transformation used in
training lives in `src/evcharge/`, and this notebook imports those modules rather than
redefining them. That is deliberate — a notebook that reimplements feature logic is how
training-serving skew starts.

Runs on the committed synthetic fixture by default, so it needs **no credentials**.
Set `SOURCE = "raw"` if you hold the licensed ACN-Data extract.

Prerequisite:
```
python -m evcharge.ingest.normalise --source sample
```

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd

from evcharge.config import get_settings, load_config

SOURCE = "sample"  # set to "raw" if you hold the licensed extract

settings = get_settings()
data_config = load_config("data")

files = glob.glob(
    str(settings.interim_dir / SOURCE / "sessions" / "**" / "*.parquet"), recursive=True
)
assert files, f"Run: python -m evcharge.ingest.normalise --source {SOURCE}"

sessions = (
    pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    .sort_values("connectionTimeLocal")
    .reset_index(drop=True)
)
sessions["year_month"] = sessions["connectionTimeLocal"].dt.strftime("%Y-%m")

print(f"{len(sessions):,} sessions | {sessions['siteID'].nunique()} sites")
print(f"{sessions['year_month'].min()} .. {sessions['year_month'].max()}")
sessions.head(3)

## 1. What a session record contains

The target is `kWhDelivered`. The critical question for every other column is *when it
becomes knowable* — a feature observable only after the session ends cannot be used by a
model that must predict at plug-in.

In [ ]:
profile = pd.DataFrame(
    {
        "dtype": sessions.dtypes.astype(str),
        "missing_pct": (sessions.isna().mean() * 100).round(1),
        "distinct": sessions.nunique(),
    }
)


def availability(column: str) -> str:
    if column in data_config["post_hoc_columns"]:
        return "POST-HOC (unusable)"
    if column in data_config["identifier_columns"]:
        return "identifier (excluded)"
    if column in data_config["plug_in_time_columns"]:
        return "plug-in time"
    return "derived"


profile["availability"] = [availability(c) for c in profile.index]
profile.sort_values(["availability", "missing_pct"])

Roughly a quarter of sessions have **no driver request at all** — the driver never used
the app. That missingness is not noise to be imputed away; whether someone declared their
intent is itself predictive, so the feature builder encodes it as `has_user_inputs` and
lets the gradient-boosted model handle the NaNs natively.

## 2. The target distribution

Strongly right-skewed, which is why the reported error metric is MAE rather than RMSE,
and why sMAPE is used instead of MAPE — sessions delivering a fraction of a kWh make
percentage error explode.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sessions["kWhDelivered"].plot.hist(bins=60, ax=axes[0], edgecolor="white")
axes[0].set_title("kWh delivered")
axes[0].set_xlabel("kWh")

sessions.boxplot(column="kWhDelivered", by="siteID", ax=axes[1])
axes[1].set_title("kWh delivered by site")
axes[1].set_xlabel("siteID")
plt.suptitle("")
plt.tight_layout()

sessions["kWhDelivered"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2)

## 3. Arrival patterns

These are workplace sites: arrivals peak sharply in the morning commute and collapse at
weekends. A sinusoidal encoding of hour is used downstream so 23:00 and 00:00 are treated
as adjacent rather than maximally distant.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sessions["connectionTimeLocal"].dt.hour.value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set_title("Arrivals by local hour")
axes[0].set_xlabel("hour")

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
counts = sessions["connectionTimeLocal"].dt.dayofweek.value_counts().sort_index()
counts.index = [days[i] for i in counts.index]
counts.plot.bar(ax=axes[1])
axes[1].set_title("Arrivals by day of week")

plt.tight_layout()

## 4. The regime change — why this dataset was chosen

This is the whole point of the project. Workplace charging collapsed in March–April 2020,
giving the drift detector a **precisely dated ground truth** to be validated against,
rather than a shift that has to be assumed.

In [ ]:
monthly = sessions.pivot_table(
    index="year_month", columns="siteID", values="sessionID", aggfunc="count"
).fillna(0)

mean_kwh = sessions.pivot_table(
    index="year_month", columns="siteID", values="kWhDelivered", aggfunc="mean"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
monthly.plot(ax=axes[0], marker="o")
axes[0].set_title("Sessions per month")
axes[0].axvspan(
    monthly.index.get_loc("2020-03") if "2020-03" in monthly.index else 0,
    monthly.index.get_loc("2020-04") if "2020-04" in monthly.index else 0,
    alpha=0.15,
    color="red",
)
mean_kwh.plot(ax=axes[1], marker="o")
axes[1].set_title("Mean kWh per session")
for ax in axes:
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()

monthly.astype(int)

Two distinct shifts happen at once, which is what makes this a genuine MLOps problem
rather than a volume anomaly:

1. **Covariate shift** — session volume collapses.
2. **Concept shift** — mean kWh per session *rises*. The people still charging during
   lockdown were drawing far more energy per visit.

A model trained before March 2020 is therefore wrong in a way that no volume metric,
uptime check or latency graph would reveal.

## 5. Relationship to the strongest predictor

The driver's declared energy request is the dominant signal — confirmed later by
permutation importance on the validation split.

In [ ]:
declared = sessions.dropna(subset=["kWhRequested"])

ax = declared.plot.scatter(
    x="kWhRequested", y="kWhDelivered", alpha=0.15, figsize=(6, 5)
)
lim = max(declared["kWhRequested"].quantile(0.99), declared["kWhDelivered"].quantile(0.99))
ax.plot([0, lim], [0, lim], "r--", linewidth=1, label="delivered = requested")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.legend()
ax.set_title("Delivered vs requested energy")

print("correlation:", declared["kWhDelivered"].corr(declared["kWhRequested"]).round(3))
print(
    "sessions delivering less than requested:",
    f"{(declared['kWhDelivered'] < declared['kWhRequested']).mean():.1%}",
)

Most sessions sit *below* the diagonal: drivers routinely ask for more than they take.
The gap between declared and delivered energy is a second drift axis worth monitoring in
V2, since it reflects changing driver behaviour independently of the fleet itself.

## 6. Data quality defects found

These were discovered here and are now enforced by `configs/contract.yaml`. They are real
defects in the live feed, not hypotheticals.

In [ ]:
contract = load_config("contract")

long_sessions = sessions[sessions["connectedHours"] > 168]
print(f"sessions connected over one week: {len(long_sessions)}")
if len(long_sessions):
    display(long_sessions[["sessionID", "siteID", "connectedHours", "kWhDelivered"]])

print("\nsiteID types after normalisation:", {type(v).__name__ for v in sessions["siteID"]})
print("clusterID types after normalisation:", {type(v).__name__ for v in sessions["clusterID"]})
print("\nquarantine rules enforced:")
for rule in contract["quarantine_rules"]:
    print(f"  - {rule['column']} {rule['op']} {rule.get('value')}: {rule['reason']}")

In the raw feed `siteID` arrives as both integer `2` and string `"0002"` (2,081 rows),
and `clusterID` is mixed-type too — which crashed the Parquet write before coercion was
added. Both are now blocking expectations.

## 7. What this means for modelling

| Observation | Consequence |
|---|---|
| Dated regime change in the data | Split **temporally**; a random split leaks across it |
| ~27% of sessions lack a driver request | Encode missingness; do not impute it away |
| `userInputs` is a list of revisions | Use only the earliest — later ones leak the future |
| `disconnectTime` is post-hoc | Excluded by the feature builder, enforced by tests |
| Target is right-skewed | Report MAE and sMAPE, not RMSE and MAPE |

Next: `python -m evcharge.pipeline --source sample`